## Vendor Management and Procurement Portal

In [4]:
import pandas as pd
import streamlit as st

In [5]:
%%writefile backend.py
import csv

Overwriting backend.py


### Columns for vendor and purchase files

In [6]:
vendor_fields = [ "vendor_id", "vendor_name", "phone", "email", "product_supplied"]
purchase_fields = ["order_id", "vendor_id", "vendor_name", "product_name", "quantity", "unit_price", "total_amount"]

### Save data in CSV file

In [7]:
def save_data(filename, data, fieldnames):

    file = None

    try:
        file = open(filename, "w", newline="", encoding="utf-8")

        writer = csv.DictWriter(file, fieldnames=fieldnames)

        writer.writeheader()

        writer.writerows(data)

    except (OSError, csv.Error, UnicodeError) as error:
        raise RuntimeError("Error saving file: " + str(error))

    finally:
        if file is not None:
            file.close()

### Load data in a CSV file

In [8]:
def load_data(filename, fieldnames):

    file = None

    try:
        file = open(filename, "r", newline="", encoding="utf-8")

        reader = csv.DictReader(file)

        if reader.fieldnames != fieldnames:
            raise ValueError("CSV column names are incorrect: " + filename)

        data = list(reader)

        return data

    except FileNotFoundError:

        save_data(filename, [], fieldnames)

        return []

    except (OSError, csv.Error, UnicodeError) as error:
        raise RuntimeError("Error reading file: " + str(error))

    finally:
        if file is not None:
            file.close()

### User Login

In [9]:
def check_login(username, password):

    if username == "admin" and password == "230806":
        return True

    return False

### Function to add a new vendor

In [10]:
def add_vendor(vendors, vendor_id, vendor_name, phone, email, product_supplied):

    vendor_id = vendor_id.strip().upper()
    vendor_name = vendor_name.strip()
    phone = phone.strip()
    email = email.strip()
    product_supplied = product_supplied.strip()

    # Here we check whether we have empty fields or not:
    if vendor_id == "" or vendor_name == "" or phone == "" or email == "" or product_supplied == "":
        raise ValueError("Please fill in all the fields.")

    # Here wecheck whether vendor ID already exists or not:
    for vendor in vendors:

        if vendor["vendor_id"] == vendor_id:
            raise ValueError("This Vendor ID already exists.")

    # Create a new vendor
    new_vendor = {
        "vendor_id": vendor_id,
        "vendor_name": vendor_name,
        "phone": phone,
        "email": email,
        "product_supplied": product_supplied
    }

    return new_vendor

### Function to search a vendor

In [11]:
def search_vendors(vendors, search_value):

    search_value = search_value.strip().lower()

    results = []

    for vendor in vendors:

        if search_value in vendor["vendor_id"].lower() or search_value in vendor["vendor_name"].lower():

            results.append(vendor)

    return results

### Function to create a purchase order

In [12]:
def create_purchase_order(vendors, purchases, vendor_id, product_name, quantity, unit_price):

    vendor_id = vendor_id.strip().upper()
    product_name = product_name.strip()

    selected_vendor = None

    # Here we have to find the selected vendors:
    for vendor in vendors:

        if vendor["vendor_id"] == vendor_id:

            selected_vendor = vendor

            break

    if selected_vendor is None:
        raise ValueError("Vendor not found.")

    if product_name == "":
        raise ValueError("Please enter a product name.")

    # Here we have to convert quantity and price into numeric values:
    try:
        quantity = int(quantity)

        unit_price = float(unit_price)

    except (ValueError, TypeError):
        raise ValueError("Please enter valid quantity and price.")

    if quantity <= 0:
        raise ValueError("Quantity must be greater than zero.")

    if unit_price <= 0:
        raise ValueError("Price must be greater than zero.")

    # Here we restrict the unique purchase order ID:
    next_number = 1

    while True:

        order_id = "PO" + str(next_number).zfill(3)

        order_found = False

        for order in purchases:

            if order["order_id"] == order_id:

                order_found = True

                break

        if order_found == False:
            break

        next_number = next_number + 1

    # Here we calculate the total amount :
    total_amount = quantity * unit_price
    
    #  Here we have to create purchase order:
    new_order = {
        "order_id": order_id,
        "vendor_id": selected_vendor["vendor_id"],
        "vendor_name": selected_vendor["vendor_name"],
        "product_name": product_name,
        "quantity": str(quantity),
        "unit_price": str(round(unit_price, 2)),
        "total_amount": str(round(total_amount, 2))
    }

    return new_order

### Function for total

In [13]:
def calculate_total_spending(purchases):

    total_spending = 0

    for order in purchases:

        total_spending = total_spending + float(order["total_amount"])

    return total_spending

### CSV files creation

In [14]:
from backend import *

vendors = load_data("vendors.csv", vendor_fields)

purchases = load_data("purchases.csv", purchase_fields)

print("Vendor data loaded successfully.")

print("Purchase data loaded successfully.")

print("Total vendors:", len(vendors))

print("Total purchase orders:", len(purchases))

Vendor data loaded successfully.
Purchase data loaded successfully.
Total vendors: 0
Total purchase orders: 0


In [15]:
df = pd.read_csv("vendors.csv")

df.columns = df.columns.str.strip().str.lower()

df.to_csv("vendors.csv", index=False)

print("Column names updated successfully!")
print(df.columns.tolist())

Column names updated successfully!
['vendor_id', 'vendor_name', 'phone', 'email', 'product_supplied']


In [16]:
df = pd.read_csv("purchases.csv")

df.columns = df.columns.str.strip().str.lower()

df.to_csv("purchases.csv", index=False)

print("Purchase column names updated successfully!")

print(df.columns.tolist())

Purchase column names updated successfully!
['order_id', 'vendor_id', 'vendor_name', 'product_name', 'quantity', 'unit_price', 'total_amount']


In [17]:
# Read vendors CSV file
df = pd.read_csv("vendors.csv")

# Convert all column names into lowercase
df.columns = df.columns.str.strip().str.lower()

# Save the updated CSV file
df.to_csv("vendors.csv", index=False)

print("Column names updated successfully!")
print(df.columns.tolist())

Column names updated successfully!
['vendor_id', 'vendor_name', 'phone', 'email', 'product_supplied']


In [ ]:
!streamlit run front.py

In [18]:
import sys
print(sys.executable)

C:\Users\Hunain Javaid\anaconda3\python.exe
